# SmolVLA — base replay + LIBERO-Mem T1/T3
Dựa trên `working_notebooks/memory-sft-nb.ipynb` đã chạy trên Kaggle. Bật **Internet**, chọn GPU (khuyến nghị **T4 ×2**), attach code và **prepared mixed shards**.

Notebook chỉ train; dữ liệu, split và memory đã được chuẩn bị trước. Không cài simulator hoặc detector. Mặc định chạy **F0/no-memory**; F1 dùng cùng sample selection và thay `CONDITION = "memory"` với dataset có predicted memory.

Pilot chuẩn bị sẵn dùng **7/10 task LIBERO-10** cho base replay và toàn bộ T1/T3; phạm vi replay này được ghi trong manifest. Ba file base còn thiếu chưa được thay thế bằng dữ liệu khác.


In [ ]:
from pathlib import Path

CODE_SOURCE = Path("/kaggle/input/datasets/luquanglinh/memory-sft-code")  # thư mục code hoặc file zip
DATA_DIR = Path("/kaggle/input/libero10base7-mem-t1t3")  # chứa manifest.json + shards/
CONDITION = "no_memory"  # đổi thành "memory" cho F1 đã có predicted-memory shards
STEPS = 3000
SEED = 42
BATCH_PER_GPU = 2  # nếu OOM: đổi thành 1; accumulation bên dưới giữ global batch 32
RUN_TRAINING = True


## 1. Thư viện
Giữ PyTorch/CUDA có sẵn của Kaggle. Hai phiên bản LeRobot/Transformers dưới đây lấy từ output của notebook reference; không tạo venv và không thay phiên bản Python/NumPy thủ công. Bỏ torchao giống notebook đã chạy để tránh integration không dùng đến.


In [ ]:
%pip uninstall -y torchao
%pip install "lerobot[smolvla,peft,dataset]==0.6.1" "transformers==5.5.4" "peft>=0.12.0" "accelerate>=0.34.0" "h5py>=3.10.0" "huggingface-hub>=0.25.0"


## 2. Code và checkpoint
Copy code sang working; script đã xử lý các tên padding mask của LeRobot, không cần patch source trong notebook. Assets giữ nguyên checkpoint/tokenizer và normalization đã pin.


In [ ]:
import json
import shutil
import subprocess
import sys
import zipfile

REPO = Path("/kaggle/working/mem-sft-code")
source = CODE_SOURCE
if source.is_dir() and (source / "memory_sft_code.zip").is_file():
    source = source / "memory_sft_code.zip"
if source.is_file():
    with zipfile.ZipFile(source) as bundle:
        bundle.extractall(REPO)
else:
    shutil.copytree(source, REPO, dirs_exist_ok=True)
ASSETS_DIR = Path("/kaggle/temp/sft-assets")
subprocess.run([sys.executable, str(REPO / "scripts/memory_sft_assets.py"),
                "fetch", str(ASSETS_DIR)], check=True, cwd=REPO)


## 3. Dataset và GPU
Prepared dataset đã khóa initial-state split 80/10/10, grouping chung T1/T3; base giữ instruction gốc. Trainer kiểm hash dữ liệu một lần trước train, không kiểm lại mỗi batch. Dùng cùng dataset cho F0/F1 khi muốn đối chứng memory; F0 tự bỏ memory text.


In [ ]:
import torch
from datetime import datetime

manifest = json.loads((DATA_DIR / "manifest.json").read_text())
if manifest.get("artifact") != "mixed_base_mem_sft_shards":
    raise ValueError("Attach mixed SFT shards, không dùng prompt index hoặc dataset legacy.")
print("Counts:", manifest["counts"])
print("Split:", manifest["split_lock_sha256"])
print("Sample selection:", manifest["sample_selection_sha256"])
print("Memory:", manifest["memory_source"], "| Base scope:", manifest["retention_scope"])
print("Base replay tasks:", len(manifest["retention_replay_tasks"]))
NUM_GPUS = torch.cuda.device_count()
if NUM_GPUS == 0:
    raise RuntimeError("Chọn GPU trong Kaggle Settings rồi chạy lại.")
if BATCH_PER_GPU < 1 or 32 % (BATCH_PER_GPU * NUM_GPUS):
    raise ValueError("BATCH_PER_GPU × NUM_GPUS phải chia hết 32 để giữ global batch.")
ACCUMULATION = 32 // (BATCH_PER_GPU * NUM_GPUS)
print("GPU:", [torch.cuda.get_device_name(i) for i in range(NUM_GPUS)])
print("Global batch:", BATCH_PER_GPU * NUM_GPUS * ACCUMULATION)
RUN_DIR = Path("/kaggle/working") / (CONDITION + "-" + datetime.now().strftime("%Y%m%d-%H%M%S"))
launch = [sys.executable, "-m", "accelerate.commands.launch"]
if NUM_GPUS > 1:
    launch += ["--multi_gpu"]
launch += ["--num_processes", str(NUM_GPUS), "--num_machines", "1",
           "--mixed_precision", "fp16", "--dynamo_backend", "no", str(REPO / "scripts/train_memory_sft.py")]
common = ["--assets-dir", str(ASSETS_DIR), "--data-dir", str(DATA_DIR),
          "--expected-gpus", str(NUM_GPUS), "--condition", CONDITION,
          "--adaptation", "expert_lora", "--rank", "16", "--lr", "1e-4",
          "--batch-size", str(BATCH_PER_GPU), "--accumulation", str(ACCUMULATION),
          "--workers", "0", "--seed", str(SEED), "--base-fraction", "0.5", "--memory-dropout", "0"]


## 4. Smoke test — 2 optimizer updates
Chạy cùng data/sampler/condition với run chính, nhưng chỉ xem hai batch validation. Smoke không phải kết quả benchmark. Run chính phía dưới luôn bắt đầu từ checkpoint gốc, không tiếp tục weights từ smoke.


In [ ]:
SMOKE_DIR = Path(str(RUN_DIR) + "-smoke")
subprocess.run(launch + common + ["--output-dir", str(SMOKE_DIR), "--steps", "2",
    "--warmup-steps", "0", "--validation-max-batches", "2", "--save-every", "1"],
    check=True, cwd=REPO)


## 5. Fine-tune
50% base / 50% Mem mỗi global optimizer batch; cân bằng task và episode trong từng domain. VLM frozen, expert LoRA, FP16. Validation loss được báo riêng base/Mem; checkpoint selection và success rate phải đo bằng rollout, không suy từ loss. Final export là checkpoint cuối, không tự gọi là best.


In [ ]:
if RUN_TRAINING:
    subprocess.run(launch + common + ["--output-dir", str(RUN_DIR), "--steps", str(STEPS),
        "--warmup-steps", "100", "--save-every", "500", "--eval-every", "500"],
        check=True, cwd=REPO)
    print("Merged checkpoint:", RUN_DIR / "pretrained_model")
    print("Run config / versions / sampler:", RUN_DIR / "run_config.json")
    print("Validation losses:", RUN_DIR / "metrics.jsonl")


## Sau khi train
Download output bằng Kaggle **Save Version → Save & Run All** / tab Output. Giữ `run_config.json`, `environment_packages.txt`, `metrics.jsonl`, checkpoints và `pretrained_model/`.

Để chạy F1 sau F0: đổi `CONDITION` thành `memory` và attach dataset có predicted memory **cùng split/sample-selection hash**. Giữ seed, steps, batch, LR và base fraction như F0. Dataset F0 chỉ có instruction sẽ được báo rõ nếu chọn nhầm F1.

Checkpoint được đánh giá ngoài Kaggle trên LIBERO base và Mem theo protocol đã khóa; training notebook không chạy simulator. Xem `docs/LIBERO_MEM_FINETUNE_DATA_PROTOCOL.md` trong code bundle để chuẩn bị dữ liệu và kiểm định.
